In [18]:
!pip install -q langchain langchain-community langchain-core langchain-groq langchain-text-splitters faiss-cpu sentence-transformers

In [6]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [19]:
text_data = """
Artificial Intelligence is the simulation of human intelligence.
Machine Learning is a subset of AI.
Deep Learning is a subset of Machine Learning.
RAG stands for Retrieval-Augmented Generation.
Groq provides fast inference for large language models.
"""

with open("data.txt", "w") as f:
    f.write(text_data)

print("✅ data.txt created")

✅ data.txt created


In [20]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load file
loader = TextLoader("data.txt")
documents = loader.load()

# Split text
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = text_splitter.split_documents(documents)

# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings)

print("✅ Vector database created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector database created


In [21]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create retriever
retriever = vectorstore.as_retriever()

# Use latest working Groq model
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

# Function to format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG system ready!")

✅ RAG system ready!


In [24]:
response = rag_chain.invoke("What is DS?")
print("Answer:", response)

Answer: Unfortunately, there is no information provided in the context about "DS".
